# Lumina-Image 2.0 BSS/BDS Scheduler Probe

This notebook mounts Google Drive, clones or updates the lightweight experiment repo, clones the official Lumina-Image 2.0 repo, runs audit, creates manifests, validates schedules, and optionally runs smoke or mini-suite inference. It does not train and does not download weights unless explicitly enabled.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

GITHUB_REPO_URL = "https://github.com/<YOUR_ORG_OR_USER>/BSS-Scheduler-Experiments.git"
BRANCH = "lumina2-bss-bds"
EXPERIMENT_REPO_ROOT = Path("/content/BSS-Scheduler-Experiments")
LUMINA_UPSTREAM_ROOT = Path("/content/Lumina-Image-2.0")
DRIVE_ROOT = Path("/content/drive/MyDrive/Colab_Projects/Lumina2-BSS-BDS")
DRIVE_WEIGHTS_ROOT = Path("/content/drive/MyDrive/ModelWeights/Lumina-Image-2.0")
EXPERIMENT_ROOT = Path("/content/Lumina2-BSS-Runs/lumina2_bss_bds_v1")
DRIVE_EXPERIMENT_ROOT = DRIVE_ROOT / "lumina2_bss_bds_v1"
SCRIPT_ROOT = EXPERIMENT_REPO_ROOT / "bss_experiments/lumina2_bss_bds_v1/scripts"

BACKEND = "native"  # use "diffusers" if native custom grid injection is unavailable
SOLVER = "euler"
DTYPE = "bfloat16"
HEIGHT = 1024
WIDTH = 1024
CFG_SCALE = 4.0
TIME_SHIFTING_FACTOR = 6.0
SEED = 0

INSTALL_DEPS = True
DOWNLOAD_WEIGHTS = False
APPLY_NATIVE_PATCH = False
RUN_SMOKE = False
RUN_MINI_SUITE = False
SYNC_DRIVE = True
CPU_OFFLOAD = False


In [ ]:
def run(cmd, cwd=None, check=True, env=None):
    cmd = [str(part) for part in cmd]
    safe = ["<HF_TOKEN>" if os.environ.get("HF_TOKEN") and part == os.environ.get("HF_TOKEN") else part for part in cmd]
    print("$", " ".join(safe))
    proc = subprocess.run(cmd, cwd=str(cwd) if cwd else None, env=env, text=True)
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed with {proc.returncode}: {' '.join(safe)}")
    return proc

def clone_or_pull(url, path, branch=None):
    path = Path(path)
    if not path.exists():
        run(["git", "clone", url, path])
    else:
        run(["git", "fetch", "--all", "--prune"], cwd=path)
    if branch:
        run(["git", "checkout", branch], cwd=path)
        run(["git", "pull", "--ff-only"], cwd=path, check=False)
    run(["git", "status", "--short"], cwd=path, check=False)


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Drive mount skipped or failed:", repr(exc))

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
clone_or_pull(GITHUB_REPO_URL, EXPERIMENT_REPO_ROOT, BRANCH)
clone_or_pull("https://github.com/Alpha-VLLM/Lumina-Image-2.0.git", LUMINA_UPSTREAM_ROOT, None)


In [ ]:
if INSTALL_DEPS:
    run([
        sys.executable, "-m", "pip", "install", "-U",
        "diffusers", "transformers", "accelerate", "safetensors", "sentencepiece",
        "huggingface_hub", "opencv-python", "pillow", "imageio", "pandas", "numpy", "matplotlib"
    ])
    req = LUMINA_UPSTREAM_ROOT / "requirements.txt"
    if req.exists():
        run([sys.executable, "-m", "pip", "install", "-r", req], check=False)


In [ ]:
def weights_ready(path):
    path = Path(path)
    native = (path / "model_args.pth").exists() and any(path.glob("consolidated.*.pth"))
    diffusers = (path / "model_index.json").exists()
    return native or diffusers

if DOWNLOAD_WEIGHTS:
    run([
        "huggingface-cli", "download", "Alpha-VLLM/Lumina-Image-2.0",
        "--local-dir", DRIVE_WEIGHTS_ROOT,
        "--local-dir-use-symlinks", "False",
    ])
elif not weights_ready(DRIVE_WEIGHTS_ROOT):
    print("Weights are missing or incomplete at", DRIVE_WEIGHTS_ROOT)
    print("Set DOWNLOAD_WEIGHTS=True only after auth/license access is ready.")
else:
    print("Using existing weights:", DRIVE_WEIGHTS_ROOT)


In [ ]:
if APPLY_NATIVE_PATCH:
    run([sys.executable, SCRIPT_ROOT / "patch_lumina_native_custom_grid.py", "--lumina_root", LUMINA_UPSTREAM_ROOT], cwd=EXPERIMENT_REPO_ROOT)


In [ ]:
run([
    sys.executable, SCRIPT_ROOT / "audit_lumina2.py",
    "--lumina_root", LUMINA_UPSTREAM_ROOT,
    "--weights_root", DRIVE_WEIGHTS_ROOT,
    "--experiment_root", EXPERIMENT_ROOT,
], cwd=EXPERIMENT_REPO_ROOT)

run([
    sys.executable, SCRIPT_ROOT / "make_manifest_lumina2_bds.py",
    "--experiment_root", EXPERIMENT_ROOT,
    "--backend", BACKEND,
    "--solver", SOLVER,
    "--height", HEIGHT,
    "--width", WIDTH,
    "--cfg_scale", CFG_SCALE,
    "--time_shifting_factor", TIME_SHIFTING_FACTOR,
    "--seed", SEED,
], cwd=EXPERIMENT_REPO_ROOT)

SMOKE_MANIFEST = EXPERIMENT_ROOT / "manifests/lumina2_smoke_manifest.csv"
MINI_MANIFEST = EXPERIMENT_ROOT / "manifests/lumina2_prompt_suite_manifest.csv"
for manifest in [SMOKE_MANIFEST, MINI_MANIFEST]:
    run([sys.executable, SCRIPT_ROOT / "validate_schedule.py", "--manifest", manifest], cwd=EXPERIMENT_REPO_ROOT)


In [ ]:
run([
    sys.executable, SCRIPT_ROOT / "scheduler_geometry_audit.py",
    "--experiment_root", EXPERIMENT_ROOT,
    "--backend", BACKEND if BACKEND in {"native", "diffusers"} else "native",
    "--time_shifting_factor", TIME_SHIFTING_FACTOR,
], cwd=EXPERIMENT_REPO_ROOT)


In [ ]:
if RUN_SMOKE:
    cmd = [
        sys.executable, SCRIPT_ROOT / "run_manifest.py",
        "--manifest", SMOKE_MANIFEST,
        "--lumina_root", LUMINA_UPSTREAM_ROOT,
        "--weights_root", DRIVE_WEIGHTS_ROOT,
        "--experiment_root", EXPERIMENT_ROOT,
        "--drive_experiment_root", DRIVE_EXPERIMENT_ROOT,
        "--resume",
        "--dtype", DTYPE,
    ]
    if SYNC_DRIVE:
        cmd.append("--sync_drive")
    if CPU_OFFLOAD:
        cmd.append("--cpu_offload")
    run(cmd, cwd=EXPERIMENT_REPO_ROOT)
    run([sys.executable, SCRIPT_ROOT / "validate_schedule.py", "--manifest", SMOKE_MANIFEST, "--require_schedule_files"], cwd=EXPERIMENT_REPO_ROOT)
    run([sys.executable, SCRIPT_ROOT / "compute_metrics_against_ref.py", "--manifest", SMOKE_MANIFEST, "--run_root", EXPERIMENT_ROOT, "--allow_missing"], cwd=EXPERIMENT_REPO_ROOT)
    run([sys.executable, SCRIPT_ROOT / "make_figures.py", "--manifest", SMOKE_MANIFEST, "--run_root", EXPERIMENT_ROOT], cwd=EXPERIMENT_REPO_ROOT)


In [ ]:
if RUN_MINI_SUITE:
    cmd = [
        sys.executable, SCRIPT_ROOT / "run_manifest.py",
        "--manifest", MINI_MANIFEST,
        "--lumina_root", LUMINA_UPSTREAM_ROOT,
        "--weights_root", DRIVE_WEIGHTS_ROOT,
        "--experiment_root", EXPERIMENT_ROOT,
        "--drive_experiment_root", DRIVE_EXPERIMENT_ROOT,
        "--resume",
        "--dtype", DTYPE,
    ]
    if SYNC_DRIVE:
        cmd.append("--sync_drive")
    if CPU_OFFLOAD:
        cmd.append("--cpu_offload")
    run(cmd, cwd=EXPERIMENT_REPO_ROOT)
    run([sys.executable, SCRIPT_ROOT / "validate_schedule.py", "--manifest", MINI_MANIFEST, "--require_schedule_files"], cwd=EXPERIMENT_REPO_ROOT)
    run([sys.executable, SCRIPT_ROOT / "compute_metrics_against_ref.py", "--manifest", MINI_MANIFEST, "--run_root", EXPERIMENT_ROOT, "--allow_missing"], cwd=EXPERIMENT_REPO_ROOT)
    run([sys.executable, SCRIPT_ROOT / "compute_bds.py", "--run_root", EXPERIMENT_ROOT], cwd=EXPERIMENT_REPO_ROOT)
    run([sys.executable, SCRIPT_ROOT / "make_figures.py", "--manifest", MINI_MANIFEST, "--run_root", EXPERIMENT_ROOT], cwd=EXPERIMENT_REPO_ROOT)


In [ ]:
run([
    sys.executable, SCRIPT_ROOT / "write_run_reports.py",
    "--run_root", EXPERIMENT_ROOT,
    "--smoke_manifest", SMOKE_MANIFEST,
    "--mini_manifest", MINI_MANIFEST,
    "--backend", BACKEND,
    "--solver", SOLVER,
], cwd=EXPERIMENT_REPO_ROOT)

print("Final report:", EXPERIMENT_ROOT / "reports/FINAL_LUMINA2_BSS_BDS_REPORT.md")
print("Drive mirror:", DRIVE_EXPERIMENT_ROOT)
